In [1]:
# import os
# import re
# from pathlib import Path
# from docling.document_converter import DocumentConverter

# # Reliable path resolution for Jupyter Notebooks
# DATA_DIR = Path(os.getcwd()).parent / "mediassist_data"

# def extract_metadata_from_text(first_page_text, folder_name):
#     """
#     Parses the raw text of the document to build structured metadata.
#     """
#     metadata = {
#         "department": folder_name.lower(), # Uses folder hierarchy (billing, clinical, etc.)
#         "document_ref": "unknown",
#         "version": "1.0",
#         "allowed_roles": ["admin"] # Default safe role
#     }
    
#     # 1. Extract Document Reference (e.g., BILL-CODE-010)
#     ref_match = re.search(r"Document ref:\s*([^\s·\n]+)", first_page_text, re.IGNORECASE)
#     if ref_match:
#         metadata["document_ref"] = ref_match.group(1).strip()
        
#     # 2. Extract Version (e.g., Version 8.0)
#     ver_match = re.search(r"Version\s*([\d.]+)", first_page_text, re.IGNORECASE)
#     if ver_match:
#         metadata["version"] = ver_match.group(1).strip()
        
#     # 3. Extract Access/Roles (e.g., Access: Billing Executives & Admin)
#     access_match = re.search(r"Access:\s*([^\n]+)", first_page_text, re.IGNORECASE)
#     if access_match:
#         raw_roles = access_match.group(1).lower()
#         # Split roles by common separators like commas, ampersands, or 'and'
#         roles_list = re.split(r",|&|\band\b", raw_roles)
#         cleaned_roles = [role.strip() for role in roles_list if role.strip()]
#         metadata["allowed_roles"] = list(set(cleaned_roles + ["admin"]))
        
#     return metadata

# def load_documents():
#     all_docs = []
    
#     # Initialize Docling Converter
#     converter = DocumentConverter()
    
#     # FIXED: Find all PDFs and Markdown files in subfolders recursively
#     all_files = []
#     for ext in ("**/*.pdf", "**/*.md"):
#         all_files.extend(DATA_DIR.glob(ext))
    
#     for doc_file in all_files:
#         print(f"Processing: {doc_file.name} ({doc_file.suffix.upper()})...")
        
#         try:
#             # Docling handles both PDFs and .md files through this single call
#             result = converter.convert(doc_file)
#             full_text = result.document.export_to_markdown()
            
#             # Extract hierarchy and metadata fields from the text layout
#             doc_metadata = extract_metadata_from_text(full_text, doc_file.parent.name)
            
#             all_docs.append(
#                 {
#                     "text": full_text,
#                     "metadata": {
#                         "source": doc_file.name,
#                         "file_type": doc_file.suffix.lower().replace(".", ""),
#                         "department": doc_metadata["department"],
#                         "document_ref": doc_metadata["document_ref"],
#                         "version": doc_metadata["version"],
#                         "allowed_roles": doc_metadata["allowed_roles"]
#                     }
#                 }
#             )
#         except Exception as e:
#             print(f"Error processing {doc_file.name}: {e}")
            
#     return all_docs

# # Run and inspect
# docs = load_documents()
# print(f"\nLoaded {len(docs)} documents successfully.")
# if docs:
#     print("\n=== EXTRACTED METADATA SAMPLE FROM LAST DOCUMENT ===")
#     print(docs[-1]["metadata"])


In [2]:
import os
import re
from pathlib import Path
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker

# Reliable path resolution for Jupyter Notebooks
DATA_DIR = Path(os.getcwd()).parent / "mediassist_data"

def extract_metadata_from_text(first_page_text, folder_name):
    """
    Parses the raw text of the document to build structured metadata.
    """
    metadata = {
        "department": folder_name.lower(),
        "document_ref": "unknown",
        "version": "1.0",
        "allowed_roles": ["admin"]
    }
    
    # 1. Extract Document Reference
    ref_match = re.search(r"Document ref:\s*([^\s·\n]+)", first_page_text, re.IGNORECASE)
    if ref_match:
        metadata["document_ref"] = ref_match.group(1).strip()
        
    # 2. Extract Version
    ver_match = re.search(r"Version\s*([\d.]+)", first_page_text, re.IGNORECASE)
    if ver_match:
        metadata["version"] = ver_match.group(1).strip()
        
    # 3. Extract Access/Roles
    access_match = re.search(r"Access:\s*([^\n]+)", first_page_text, re.IGNORECASE)
    if access_match:
        raw_roles = access_match.group(1).lower()
        roles_list = re.split(r",|&|\band\b", raw_roles)
        cleaned_roles = [role.strip() for role in roles_list if role.strip()]
        metadata["allowed_roles"] = list(set(cleaned_roles + ["admin"]))
        
    return metadata

def load_and_chunk_documents():
    all_chunks = []
    
    # Initialize Docling Converter
    converter = DocumentConverter()
    
    # Pass 1 & 2 Config: Structural chunker with token limits
    chunker = HybridChunker(
        tokenizer="sentence-transformers/all-MiniLM-L6-v2", 
        max_tokens=256,                                      
        merge_peers=True                                     
    )
    
    # Find all PDFs and Markdown files recursively
    all_files = []
    for ext in ("**/*.pdf", "**/*.md"):
        all_files.extend(DATA_DIR.glob(ext))
    
    for doc_file in all_files:
        print(f"Processing & Chunking: {doc_file.name} ({doc_file.suffix.upper()})...")
        
        try:
            # Pass 1: Parse layout into a structural tree
            result = converter.convert(doc_file)
            doc_structure = result.document
            
            # Extract parent metadata
            full_text = doc_structure.export_to_markdown()
            doc_metadata = extract_metadata_from_text(full_text, doc_file.parent.name)
            
            # Pass 2: Apply token-aware chunk boundaries over layout objects
            doc_chunks = chunker.chunk(doc_structure)
            
            for i, chunk in enumerate(doc_chunks, start=1):
                # FIXED: Restored serialize() which matches your package environment version
                chunk_text = chunker.serialize(chunk)
                
                # FIXED: Safely check if headings are objects with a .text attribute or plain strings
                heading_list = []
                if chunk.meta.headings:
                    for node in chunk.meta.headings:
                        if hasattr(node, "text"):
                            heading_list.append(node.text)
                        else:
                            heading_list.append(str(node))

                all_chunks.append(
                    {
                        "text": chunk_text,
                        "metadata": {
                            "source": doc_file.name,
                            "file_type": doc_file.suffix.lower().replace(".", ""),
                            "chunk_id": f"{doc_file.stem}_chunk_{i}",
                            "department": doc_metadata["department"],
                            "document_ref": doc_metadata["document_ref"],
                            "version": doc_metadata["version"],
                            "allowed_roles": doc_metadata["allowed_roles"],
                            "heading_hierarchy": heading_list
                        }
                    }
                )
        except Exception as e:
            print(f"Error processing {doc_file.name}: {e}")
            
    return all_chunks

# Run the fixed pipeline
chunks = load_and_chunk_documents()
print(f"\nSuccessfully generated {len(chunks)} structural chunks.")

if chunks:
    print("\n=== SAMPLE CHUNK TEXT FROM FIRST CHUNK ===")
    print(chunks[0]["text"])
    print("\n=== SAMPLE CHUNK METADATA ===")
    import pprint
    pprint.pprint(chunks[0]["metadata"])



Processing & Chunking: billing_codes.pdf (.PDF)...


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-06-16 21:42:45,777 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-16 21:42:45,798 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\rames\anaconda3\envs\ai_env\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-16 21:42:45,799 [RapidOCR] main.py:57: Using C:\Users\rames\anaconda3\envs\ai_env\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-16 21:42:45,929 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-16 21:42:45,934 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\rames\anaconda3\envs\ai_env\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-16 21:42:45,935 [RapidOCR] main.py:57

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1453 > 512). Running this sequence through the model will result in indexing errors
C:\Users\rames\AppData\Local\Temp\ipykernel_20308\302047049.py:76: DeprecationWarning: Use contextualize() instead.
  chunk_text = chunker.serialize(chunk)


Processing & Chunking: diagnostic_reference.pdf (.PDF)...


...

C:\Users\rames\AppData\Local\Temp\ipykernel_20308\302047049.py:76: DeprecationWarning: Use contextualize() instead.
  chunk_text = chunker.serialize(chunk)


Processing & Chunking: infection_control.pdf (.PDF)...


C:\Users\rames\AppData\Local\Temp\ipykernel_20308\302047049.py:76: DeprecationWarning: Use contextualize() instead.
  chunk_text = chunker.serialize(chunk)


Processing & Chunking: claim_submission_guide.md (.MD)...

Successfully generated 275 structural chunks.

=== SAMPLE CHUNK TEXT FROM FIRST CHUNK ===
Insurance Billing Code Reference
ICD-10 Diagnosis Codes, Procedure Codes & Insurer Package Rates
MediAssist Health Network Central Billing Office Document ref: BILL-CODE-010 · Version 8.0 Access: Billing Executives & Admin

=== SAMPLE CHUNK METADATA ===
{'allowed_roles': ['amp; admin', 'admin', 'billing executives'],
 'chunk_id': 'billing_codes_chunk_1',
 'department': 'billing',
 'document_ref': 'BILL-CODE-010',
 'file_type': 'pdf',
 'heading_hierarchy': ['Insurance Billing Code Reference'],
 'source': 'billing_codes.pdf',
 'version': '8.0'}


C:\Users\rames\AppData\Local\Temp\ipykernel_20308\302047049.py:76: DeprecationWarning: Use contextualize() instead.
  chunk_text = chunker.serialize(chunk)


In [3]:
#import sys
#!{sys.executable} -m pip install langchain-qdrant langchain-huggingface langchain-core


### Step 1: Ingestion with Dense + Sparse Indexing

In [7]:
import os
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
# FIXED: Imported RetrievalMode alongside FastEmbedSparse
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from langchain_core.documents import Document

# Define your model name variable
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# 1. Initialize LangChain Embedding Models
dense_embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"}, 
    encode_kwargs={"normalize_embeddings": True}
)

sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25", batch_size=32)

print("Dense embedding model:", EMBED_MODEL)
print("Sparse embedding model: Qdrant/BM25")

# 2. Convert your existing chunks into LangChain Document Objects
langchain_docs = []

for chunk in chunks:
    # Clean up HTML 'amp;' security tags from your roles list
    cleaned_roles = [role.replace("amp;", "").strip() for role in chunk["metadata"]["allowed_roles"]]
    cleaned_roles = list(set([r for r in cleaned_roles if r]))
    
    # REQUIREMENT 1: Embed text with its parent section heading context prefixed
    hierarchy = chunk["metadata"]["heading_hierarchy"]
    context_prefix = f"Context: {' > '.join(hierarchy)}\n\n" if hierarchy else ""
    enriched_text = f"{context_prefix}{chunk['text']}"
    
    # REQUIREMENT 2: Map the complete required metadata schema
    metadata_schema = {
        "source": chunk["metadata"]["source"],
        "file_type": chunk["metadata"]["file_type"],
        "chunk_id": chunk["metadata"]["chunk_id"],
        "department": chunk["metadata"]["department"],
        "document_ref": chunk["metadata"]["document_ref"],
        "version": str(chunk["metadata"]["version"]),
        "allowed_roles": cleaned_roles,  # Kept as a clean list for LangChain matching
        "heading_hierarchy": hierarchy
    }
    
    # Build LangChain Core Document
    langchain_docs.append(
        Document(page_content=enriched_text, metadata=metadata_schema)
    )

# 3. Create the Persistent Hybrid Collection
# FIXED: Used RetrievalMode.HYBRID instead of QdrantVectorStore.HYBRID
db = QdrantVectorStore.from_documents(
    documents=langchain_docs,
    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,
    path="./mediassist_langchain_hybrid_db",
    collection_name="mediassist_hybrid_rag",
    retrieval_mode=RetrievalMode.HYBRID # Correctly activates unified dense + sparse tracking
)

print(f"\nSuccessfully indexed {len(langchain_docs)} documents into LangChain Hybrid Vector Store.")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Dense embedding model: sentence-transformers/all-MiniLM-L6-v2
Sparse embedding model: Qdrant/BM25

Successfully indexed 275 documents into LangChain Hybrid Vector Store.


In [15]:
from qdrant_client.models import Filter, FieldCondition, MatchAny

def secure_langchain_hybrid_search(user_query, user_role, top_k=3):
    """
    Executes a single hybrid semantic + keyword lookup query.
    Enforces adaptive role-based mapping filters to match extracted index strings.
    """
    role_to_check = user_role.lower().strip()
    
    # Map user inputs directly to your database's exact extracted string variations
    role_map = {
        "nurse": ["nurse", "nurses", "nurse staff", "all"],
        "doctor": ["doctor", "doctors", "clinical", "all"],
        "admin": ["admin", "`admin`", "all"],
        "billing": ["billing", "billing executives", "`billing_executive`", "all"]
    }
    
    # Get the target search list based on user input keywords
    allowed_variants = [role_to_check]
    for key, variants in role_map.items():
        if key in role_to_check:
            allowed_variants.extend(variants)
            
    # Force unique values
    allowed_variants = list(set(allowed_variants))
        
    # Execute the MatchAny filter condition inside the Qdrant engine
    security_filter = Filter(
        must=[
            FieldCondition(
                key="metadata.allowed_roles",
                match=MatchAny(any=allowed_variants)
            )
        ]
    )
    
    results = db.similarity_search(
        query=user_query,
        k=top_k,
        filter=security_filter
    )
    
    return results





In [17]:
query = "What is the correct IV cannula size for a paediatric patient under 5kg?"
user_role = "nurse"

print(f"Executing secure hybrid search for: '{query}' with role validation for: '{user_role}'\n")

matched_docs = secure_langchain_hybrid_search(query, user_role)

if not matched_docs:
    print("Zero authorized matches discovered. Verify collection population steps.")
else:
    for i, doc in enumerate(matched_docs, start=1):
        print(f"--- Fused Hybrid Result Rank {i} ---")
        print(f"Source: {doc.metadata.get('source', 'unknown')} | ID: {doc.metadata.get('chunk_id', 'unknown')}")
        print(f"Stored Access Roles: {doc.metadata.get('allowed_roles')}")
        print(f"Text Contents:\n{doc.page_content}")
        print("-" * 50)



Executing secure hybrid search for: 'What is the correct IV cannula size for a paediatric patient under 5kg?' with role validation for: 'nurse'

--- Fused Hybrid Result Rank 1 ---
Source: icu_nursing_procedures.pdf | ID: icu_nursing_procedures_chunk_14
Stored Access Roles: ['doctors', 'nurses', 'admin']
Text Contents:
Context: Cannula sizing by indication

Cannula sizing by indication
Blood transfusion, Recommended Gauge = ≥ 18G. Rapid fluid resuscitation, Recommended Gauge = ≥ 16G. Routine medication, Recommended Gauge = 20-22G. Paediatric < 5 kg, Recommended Gauge = 24G. Paediatric 5-20 kg, Recommended Gauge = 22G. Paediatric > 20 kg, Recommended Gauge = 20G
--------------------------------------------------
--- Fused Hybrid Result Rank 2 ---
Source: icu_nursing_procedures.pdf | ID: icu_nursing_procedures_chunk_10
Stored Access Roles: ['doctors', 'nurses', 'admin']
Text Contents:
Context: Sizing

Sizing
Adult: 14-16 Fr . Paediatric: select by age-appropriate formula.
----------------